## 60_verify_models  (NEW workspace) — Model Artifacts Migration step 3 of 3

Acceptance test for the 5 migrated models (video's test_parity practice):
  - loads each model @latest from the UC Model Registry
  - scores the same 5 training-table rows used in the old workspace
  - compares against parity_reference.csv exported from the OLD workspace
PASS = model loads + finite predictions; PARITY = informational match.
Widgets: CATALOG/SCHEMA/VOLUME locate the unpacked export (same as 50).
Bootstrap (old-school, production shape): setup_wd_path -> prepare_requirements
  -> %pip install -r "$REQ_PATH" -q   (see runbook Appendix A.4)


In [ ]:

%run ./lib/config_lib


In [ ]:

############################
# set working dir path + stage requirements (old-school bootstrap)
############################
import os

PROJECT_PATH = setup_wd_path(dbutils)
REQ_PATH = prepare_requirements(PROJECT_PATH)
print(f"PROJECT_PATH : {PROJECT_PATH}")
print(f"REQ_PATH     : {REQ_PATH}")


In [ ]:

%pip install -r "$REQ_PATH" -q


In [ ]:

dbutils.widgets.text("CATALOG", "practice_ml", "Landing-zone catalog")
dbutils.widgets.text("SCHEMA", "models", "Landing-zone schema")
dbutils.widgets.text("VOLUME", "staging_artifacts", "Landing-zone volume")

CATALOG = dbutils.widgets.get("CATALOG").strip()
SCHEMA = dbutils.widgets.get("SCHEMA").strip()
VOLUME = dbutils.widgets.get("VOLUME").strip()
UNPACK_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/models_export"


In [ ]:

import json
import os
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

assert os.path.exists(f"{UNPACK_DIR}/parity_reference.csv"), "models_export not unpacked — run 50 first"
assert os.path.exists(f"{UNPACK_DIR}/parity_features.json"), (
    "parity_features.json missing — re-export with the updated 30_export_models"
)

reference = pd.read_csv(f"{UNPACK_DIR}/parity_reference.csv")
with open(f"{UNPACK_DIR}/parity_features.json") as f:
    parity_feats = json.load(f)
X_parity = np.array(parity_feats["rows"], dtype=float)

client = MlflowClient()
pred_cols = [c for c in reference.columns if c.startswith("pred_")]
rows = []

for _, ref in reference.iterrows():
    simple = ref["simple_name"]
    full_name = f"{CATALOG}.{SCHEMA}.{simple}"
    status, parity, detail = "FAIL", "-", ""
    try:
        m = mlflow.pyfunc.load_model(f"models:/{full_name}@latest")
        preds = np.asarray(m.predict(X_parity), dtype=float).ravel()
        assert len(preds) == len(pred_cols) and np.isfinite(preds).all()
        status = "PASS"
        expected = ref[[c for c in pred_cols]].to_numpy(dtype=float)
        if np.allclose(preds, expected, rtol=1e-6, atol=1e-6):
            parity = "MATCH"
        else:
            parity = "DIFF"
            detail = f"max_abs_diff={np.max(np.abs(preds - expected)):.3e} (check sklearn versions)"
    except Exception as e:
        detail = str(e)[:160]
    rows.append({"model": full_name, "load_score": status, "parity_vs_old": parity, "note": detail})

print(f"MLflow version: {mlflow.__version__}")
display(pd.DataFrame(rows))
assert all(r["load_score"] == "PASS" for r in rows), "At least one model failed to load/score"


In [ ]:

print("=== REGISTERED MODELS IN UNITY CATALOG ===")
for rm in client.search_registered_models(f"name LIKE '{CATALOG}.{SCHEMA}.%'"):
    aliases = rm.aliases if hasattr(rm, "aliases") else {}
    versions = [v.version for v in client.search_model_versions(f"name='{rm.name}'")]
    print(f"  {rm.name}  versions={versions}  aliases={aliases}")
print("\nAll 5 migrated models verified. Update MIGRATION_CHECKLIST.md rows M1-M5.")
